# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd
# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(metadata['name'] + ": " + metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities including record sets, fields, and columns are referenced by their `@id` according to Croissant schema best practices.

In [ ]:
# List available record sets by @idrecord_sets = dataset.metadata.record_sets
print("Record Sets (@id):")for record_set in record_sets:    print(f"- {record_set['@id']}: {record_set.get('name', '')}")
# For each record set, list its fields and their @idfor record_set in record_sets:    print(f"\nRecord Set: {record_set['@id']} ({record_set.get('name', '')})")    if 'fields' in record_set:        print("  Fields (@id):")        for field in record_set['fields']:            print(f"    - {field['@id']} (name: {field.get('name', '')})")    if 'columns' in record_set:        print("  Columns (@id):")        for col in record_set['columns']:            print(f"    - {col['@id']} (name: {col.get('name', '')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Below, we use each record set's `@id` and field/column `@id`s as discovered above. For demonstration, extraction is performed for all available record sets.

In [ ]:
# Extract data from each record set
# Get record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Record Set '{record_set_id}' columns:")
    print(dataframes[record_set_id].columns.tolist())
    if not dataframes[record_set_id].empty:
        display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalizing, and grouping. We'll select a numeric field by its `@id` and demonstrate common EDA operations.

Ensure that all references to record sets and fields use their `@id`.

In [ ]:
# For demonstration, pick the first record set with at least one numeric field.
import numpy as np

chosen_record_set_id = None
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    rs_id = rs['@id']
    df = dataframes[rs_id]
    # Find a numeric field from metadata
    for f in rs.get('fields', []) + rs.get('columns', []):
        # Try to select a numeric type
        if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
            field_id = f['@id']
            if field_id in df.columns:
                numeric_field_id = field_id
                chosen_record_set_id = rs_id
                # Optionally, select group field
                for g in rs.get('fields', []) + rs.get('columns', []):
                    if g.get('dataType', '').lower() == 'text' and g['@id'] != numeric_field_id:
                        group_field_id = g['@id']
                        break
                break
    if numeric_field_id:
        break

if chosen_record_set_id and numeric_field_id:
    df = dataframes[chosen_record_set_id]
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records from '{chosen_record_set_id}' with '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by text field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field and relationships between fields, referenced by their `@id`. Plot histograms and group-wise comparisons if applicable.

In [ ]:
import matplotlib.pyplot as plt

if chosen_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    dataframes[chosen_record_set_id][numeric_field_id].hist(bins=15)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f"Histogram of '{numeric_field_id}' in '{chosen_record_set_id}' record set")
    plt.show()

    # If grouping is available
    if group_field_id:
        plt.figure(figsize=(10,5))
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load, explore, and process tabular clinical and molecular data about cancer survivors with second primary colorectal cancer. All entities and fields were referenced by their `@id` for compatibility with Croissant schema conventions.

- We reviewed available record sets, fields, and columns.
- Extracted data into DataFrames using their unique IDs.
- Performed basic EDA, such as filtering and normalization.
- Visualized numeric distribution and group-wise comparisons.

This workflow supports reproducible clinical and scientific analyses, ensuring schema traceability and FAIR dataset compliance.